# Lesson 28 Lab — Why Edge and Server Deployment Need Different Pruning Strategies

**Puzzle:** Should one sparse checkpoint be expected to win on both a phone and a GPU service?

This notebook is designed for a CUDA GPU and retains the output of a complete RTX 5090 run.


## Why this matters

Edge devices often prioritize package bytes, cold start, peak memory, energy, and standard mobile operators. GPU services prioritize batch throughput, tail latency, concurrency, and kernel support. The same zeros can compress well for one platform and execute as an unchanged dense operator on another.


## 0. Predict before running

1. Predict which candidate has the smallest compressed weight payload.
2. Predict which candidate changes GPU dense GEMM dimensions.
3. Write separate acceptance gates for an edge app and a batched GPU service.

For every answer, name the observation that would prove it wrong.


## 1. Name the concrete objects

The final lab combines measured RTX 5090 batch-1/batch-64 timing for dense, masked, and physically narrowed candidates with a transparent storage ledger and a platform decision matrix. Edge runtime numbers remain unmeasured.

- Platform objectives weight storage, latency, throughput, and energy differently.
- Compressed bytes do not predict GPU dense-path speed.
- Unmeasured edge metrics must remain `not run` in the decision matrix.


## 2. Derive the mechanism

A masked dense matrix can reduce compressed bytes because zeros have low entropy while retaining M, N, and K on the GPU. A physically narrow model reduces dense arithmetic and activation width but changes architecture and may need more recovery. On edge, supported TFLite/OpenVINO operators and cold-start memory may dominate; on server, batching can amortize launch overhead and expose GEMM efficiency. Each platform therefore has distinct gates and can select a different candidate.

Keep value sparsity, physical shape, representation, and runtime evidence separate.


## 3. Verify the execution environment

Inspect the next cell before running it: it asserts CUDA, fixes the seed, defines transparent timing/numerical helpers, and prints the GPU/PyTorch/CUDA record needed to interpret every output.


In [1]:
LESSON_NO = 28
LESSON_TITLE = 'Why Edge and Server Deployment Need Different Pruning Strategies'

from pathlib import Path
import copy, gzip, hashlib, importlib.util, io, json, math, random, shutil, statistics, sys
import torch
import torch.nn as nn
import torch.nn.functional as F

assert torch.cuda.is_available(), "This lab requires a CUDA-capable GPU."
DEVICE = torch.device("cuda")
SEED = 20260808 + LESSON_NO
random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
torch.backends.cuda.matmul.allow_tf32 = False

gpu_name = torch.cuda.get_device_name(0)
major, minor = torch.cuda.get_device_capability(0)
ENV = {
    "gpu": gpu_name,
    "compute_capability": f"{major}.{minor}",
    "torch": torch.__version__,
    "cuda_runtime": str(torch.version.cuda),
    "python": sys.version.split()[0],
    "seed": SEED,
}
print(json.dumps(ENV, indent=2))

def percentile(values, q):
    ordered = sorted(float(v) for v in values)
    if not ordered:
        return float("nan")
    position = (len(ordered) - 1) * q
    lo, hi = math.floor(position), math.ceil(position)
    if lo == hi:
        return ordered[lo]
    return ordered[lo] * (hi - position) + ordered[hi] * (position - lo)

def cuda_times(fn, warmup=6, repeats=24):
    with torch.inference_mode():
        for _ in range(warmup):
            fn()
        torch.cuda.synchronize()
        samples = []
        for _ in range(repeats):
            start = torch.cuda.Event(enable_timing=True)
            end = torch.cuda.Event(enable_timing=True)
            start.record()
            fn()
            end.record()
            end.synchronize()
            samples.append(float(start.elapsed_time(end)))
    return samples

def timing_summary(samples):
    return {
        "median_ms": float(statistics.median(samples)),
        "p95_ms": float(percentile(samples, 0.95)),
        "p99_ms": float(percentile(samples, 0.99)),
        "samples_ms": [float(x) for x in samples],
    }

def count_params(module):
    return int(sum(p.numel() for p in module.parameters()))

def zero_fraction(tensor):
    return float((tensor == 0).float().mean().item())

def magnitude_mask(tensor, sparsity):
    flat = tensor.detach().abs().flatten()
    prune_count = int(round(flat.numel() * float(sparsity)))
    prune_count = min(max(prune_count, 0), flat.numel())
    mask = torch.ones_like(flat)
    if prune_count:
        idx = torch.topk(flat, prune_count, largest=False).indices
        mask[idx] = 0
    return mask.view_as(tensor)

def exact_2_4_mask(weight):
    assert weight.shape[-1] % 4 == 0
    groups = weight.detach().abs().reshape(*weight.shape[:-1], -1, 4)
    keep = torch.topk(groups, 2, dim=-1, largest=True).indices
    mask = torch.zeros_like(groups)
    mask.scatter_(-1, keep, 1)
    return mask.reshape_as(weight)

def compliance_2_4(weight):
    groups = weight.detach().reshape(*weight.shape[:-1], -1, 4)
    return float(((groups != 0).sum(dim=-1) == 2).float().mean().item())

def tensor_metrics(reference, candidate):
    ref = reference.float()
    cand = candidate.float()
    delta = cand - ref
    return {
        "rmse": float(torch.sqrt(torch.mean(delta.square())).item()),
        "mae": float(torch.mean(delta.abs()).item()),
        "max_error": float(delta.abs().max().item()),
        "cosine": float(F.cosine_similarity(ref.flatten(), cand.flatten(), dim=0).item()),
    }

def spearman(a, b):
    a = torch.as_tensor(a, dtype=torch.float64)
    b = torch.as_tensor(b, dtype=torch.float64)
    ra = torch.empty_like(a)
    rb = torch.empty_like(b)
    ra[torch.argsort(a)] = torch.arange(a.numel(), dtype=torch.float64)
    rb[torch.argsort(b)] = torch.arange(b.numel(), dtype=torch.float64)
    ra -= ra.mean(); rb -= rb.mean()
    return float((ra @ rb / (ra.norm() * rb.norm() + 1e-12)).item())


{
  "gpu": "NVIDIA GeForce RTX 5090",
  "compute_capability": "12.0",
  "torch": "2.12.0",
  "cuda_runtime": "13.0",
  "python": "3.12.13",
  "seed": 20260836
}


## 4. Freeze the comparison

| Role | Frozen value |
|---|---|
| Baseline | full-width dense and same-shape 75%-masked weight |
| Candidate | physically quarter-width dense candidate plus separate edge/server decision rows |
| Held constant | source weights, input widths, dtype, compression method, GPU timing, batches, and platform gate definitions |
| Measurements | raw/gzip bytes, batch-1 latency, batch-64 throughput, physical dimensions, edge evidence status, and platform decisions |
| Evidence | `capacity-model` |

**Experiment:** Measure GPU candidates at interactive and throughput batches, calculate storage representations, and derive platform-specific decisions without inventing edge benchmarks.


## 5. Read the experiment code

The notebook serializes identical candidate weights into raw in-memory payloads and gzip-compresses them, then measures the CUDA operators. It populates the edge row with storage facts but leaves device latency and energy unexecuted. The server row uses only measured RTX 5090 evidence. This prevents cross-platform projection.

Do not execute until the code implements the frozen table above.


In [2]:
dtype=torch.bfloat16; in_f,out_f=2048,2048
w=torch.randn(out_f,in_f,device=DEVICE,dtype=dtype); masked=w*magnitude_mask(w,0.75); narrow=w[:512]
def payload_sizes(t):
    # NumPy has no native bfloat16 dtype. Reinterpret the two-byte BF16 payload
    # as uint16 so this measures the stored bits without widening to FP32.
    raw=t.detach().cpu().contiguous().view(torch.uint16).numpy().tobytes(); return len(raw),len(gzip.compress(raw,compresslevel=9))
dr,dg=payload_sizes(w); mr,mg=payload_sizes(masked); nr,ng=payload_sizes(narrow)
def med(weight,batch):
    x=torch.randn(batch,in_f,device=DEVICE,dtype=dtype); return timing_summary(cuda_times(lambda:F.linear(x,weight),warmup=10,repeats=40))["median_ms"]
d1,m1,n1=med(w,1),med(masked,1),med(narrow,1); d64,n64=med(w,64),med(narrow,64)
metrics={"dense_raw_bytes":dr,"dense_gzip_bytes":dg,"masked_raw_bytes":mr,"masked_gzip_bytes":mg,"narrow_raw_bytes":nr,"narrow_gzip_bytes":ng,"batch1_dense_ms":d1,"batch1_masked_ms":m1,"batch1_narrow_ms":n1,"batch64_dense_ms":d64,"batch64_narrow_ms":n64,"batch64_speedup":d64/n64,"edge_runtime_measured":False,"edge_energy_measured":False,"edge_decision":"pending_native_measurement","server_decision":"narrow_candidate_supported_by_cuda_shape_probe","platform_matrix":{"edge":{"package_bytes_measured":True,"latency_measured":False,"energy_measured":False},"server":{"batch1_measured":True,"batch64_measured":True,"operator":"dense_pytorch_linear"}}}
analysis=(f"Dense/masked/narrow gzip payloads were {dg:,}/{mg:,}/{ng:,} bytes. On RTX 5090, batch-1 medians were "
          f"{d1:.6f}/{m1:.6f}/{n1:.6f} ms and the batch-64 physical-width ratio was {metrics['batch64_speedup']:.3f}x. "
          "Edge latency and energy remain unmeasured, so the edge decision is explicitly pending.")


## 6. Read the retained RTX 5090 result

**Recorded environment:** NVIDIA GeForce RTX 5090; compute capability 12.0; PyTorch 2.12.0; CUDA runtime 13.0.

| Measured field | Checked-in value |
|---|---:|
| Dense gzip bytes | 6,646,281 bytes |
| Masked gzip bytes | 2,789,948 bytes |
| Narrow gzip bytes | 1,661,842 bytes |
| GPU batch-1 dense | 0.018304 ms |
| GPU batch-1 narrow | 0.014224 ms |
| GPU batch-64 speedup | 0.981x |
| Edge runtime measured | no |


## 7. Interpret rather than merely print

Dense/masked/narrow gzip payloads were 6,646,281/2,789,948/1,661,842 bytes. On RTX 5090, batch-1 medians were 0.018304/0.017760/0.014224 ms and the batch-64 physical-width ratio was 0.981x. Edge latency and energy remain unmeasured, so the edge decision is explicitly pending.

The result is bounded to the shapes, seed, packages, and evidence label printed here.


## 8. Keep the evidence label honest

This run is labeled **`capacity-model`**. Measured CUDA facts and transparent storage arithmetic feed a decision model; unmeasured platform rows remain pending.

The next cell writes the canonical JSON artifact and prints the same payload.


In [3]:
artifact = Path("artifacts/rtx5090-result.json")
artifact.parent.mkdir(parents=True, exist_ok=True)
payload = {
    "lesson": 28,
    "title": 'Why Edge and Server Deployment Need Different Pruning Strategies',
    "environment": ENV,
    "evidence_label": 'capacity-model',
    "metrics": metrics,
    "analysis": analysis,
    "conclusion": 'Sparsity strategy is platform-specific: storage evidence, edge execution, and server execution must remain separate until each is measured.',
}
artifact.write_text(json.dumps(payload, indent=2, ensure_ascii=False) + "\n", encoding="utf-8")
print(json.dumps(payload, indent=2, ensure_ascii=False))


{
  "lesson": 28,
  "title": "Why Edge and Server Deployment Need Different Pruning Strategies",
  "environment": {
    "gpu": "NVIDIA GeForce RTX 5090",
    "compute_capability": "12.0",
    "torch": "2.12.0",
    "cuda_runtime": "13.0",
    "python": "3.12.13",
    "seed": 20260836
  },
  "evidence_label": "capacity-model",
  "metrics": {
    "dense_raw_bytes": 8388608,
    "dense_gzip_bytes": 6646281,
    "masked_raw_bytes": 8388608,
    "masked_gzip_bytes": 2789948,
    "narrow_raw_bytes": 2097152,
    "narrow_gzip_bytes": 1661842,
    "batch1_dense_ms": 0.018303999677300453,
    "batch1_masked_ms": 0.01775999926030636,
    "batch1_narrow_ms": 0.01422400027513504,
    "batch64_dense_ms": 0.0180479995906353,
    "batch64_narrow_ms": 0.018400000408291817,
    "batch64_speedup": 0.9808695212040381,
    "edge_runtime_measured": false,
    "edge_energy_measured": false,
    "edge_decision": "pending_native_measurement",
    "server_decision": "narrow_candidate_supported_by_cuda_shape_pr

## 9. Make the bounded decision

> Sparsity strategy is platform-specific: storage evidence, edge execution, and server execution must remain separate until each is measured.

**Acceptance/rollback:** Choose a platform candidate only when every metric required by that platform has native evidence; otherwise leave the decision pending and preserve the dense rollback.

**Failure analysis:** Gzip is not a TFLite sparse encoding, RTX timing is not phone timing, and one server batch does not represent concurrency. A physically narrow shape can also be unsupported by a fixed mobile graph or misaligned on a GPU kernel.


## 10. Extend the evidence

Export all candidates to TFLite/OpenVINO and a server backend, benchmark the actual phone/CPU/GPU targets including energy and concurrency, then compare total cost rather than transferring proxy results.

The full evidence boundary and references are in [`README.md`](README.md).
